In [ ]:
# %%
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/manipulator')

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
from pinocchio import rpy
from core import FixedBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer, DataLogger, plot_helpers
from control import TaskSpaceController, TrajectoryGenerator, OSQP_Solver

%load_ext autoreload
%autoreload 2

MJCF = '/home/frlab/mj_opt/models/scenes/ur5e/scene.xml'
URDF = '/home/frlab/mj_opt/models/ur5e/urdf/ur5e.urdf'
PKG  = '/home/frlab/mj_opt/models/ur5e'

orchestrator 구성

In [ ]:
class RobotOrchestrator:
    def __init__(self, kernel, wrapper, controller, traj, solver, dt, logger=None):
        self.kernel = kernel
        self.wrapper = wrapper
        self.controller = controller
        self.traj = traj
        self.solver = solver
        self.dt = dt
        self.logger = logger

        self.target_pos = np.zeros(3)
        self.target_R = np.eye(3)
        self.target_twist = np.zeros(6)

        self.current_ee_pos = np.zeros(3)
        self.current_ee_R = np.eye(3)
        self.current_twist = np.zeros(6)

        self.ctrl_pos = np.zeros(wrapper.nv)

        # 진단: QP vs closed-form DLS 비교
        self._diag_step  = 0
        self._diag_diffs = []

        # 관절 한계 — Pinocchio model에서 직접
        self.q_lower = wrapper.model.lowerPositionLimit
        self.q_upper = wrapper.model.upperPositionLimit
        self.dq_max  = wrapper.model.velocityLimit

    def on_control(self, t):
        # 1. 상태 업데이트
        self.kernel.update_robot_state(state)
        self.wrapper.update_model(state.q, state.dq)

        # 2. 궤적 평가
        interpolated_oM_ee, target_twist_new = self.traj.evaluate(t)
        self.target_pos[:] = interpolated_oM_ee.translation
        self.target_R[:] = interpolated_oM_ee.rotation
        self.target_twist[:] = target_twist_new

        # 3. 현재 EE 상태
        oMf, v_lin, v_ang = self.wrapper.compute_ee_state_W("ee")
        self.current_ee_pos[:] = oMf.translation
        self.current_ee_R[:] = oMf.rotation
        self.current_twist[:3] = v_lin
        self.current_twist[3:] = v_ang

        # 4. 제어 연산 (DLS QP)
        pose_err, twist_err = self.controller.compute_error(
            self.current_ee_pos, self.current_ee_R, self.current_twist,
            self.target_pos, self.target_R, self.target_twist
        )
        J_ee = self.wrapper.compute_J_W("ee")
        H, q_cost, _ = self.controller.compute_QP(pose_err, self.target_twist, J_ee)

        # 4-1. 제약: dq_max + q-range 1-step 예측
        q_curr = state.q
        l = np.maximum(-self.dq_max, (self.q_lower - q_curr) / self.dt)
        u = np.minimum( self.dq_max, (self.q_upper - q_curr) / self.dt)

        # 4-2. OSQP partial update + warm-start
        self.solver.update(H=H, q=q_cost, l=l, u=u)
        dq_des = self.solver.solve()
        if dq_des is None:
            dq_des = np.zeros(self.wrapper.nv)

        # ── 진단: closed-form DLS 와 비교
        dq_cls = self.controller.compute_DLS(pose_err, self.target_twist, J_ee)
        diff   = np.linalg.norm(dq_des - dq_cls)
        self._diag_diffs.append(diff)
        # 100 step 마다 한 번 print
        if self._diag_step % 100 == 0:
            active_lo = int((dq_cls < l + 1e-9).sum())
            active_hi = int((dq_cls > u - 1e-9).sum())
            print(f"t={t:5.2f}  ||dq_qp - dq_cls||={diff:.3e}  "
                  f"||dq_qp||={np.linalg.norm(dq_des):.3f}  "
                  f"active_lo={active_lo} active_hi={active_hi}")
        self._diag_step += 1

        # 4-3. dq_des → 1-step 적분 → 위치 명령
        q_des = self.kernel.q_mj + dq_des * self.dt
        self.kernel.ctrl_pos = q_des

        # 5. NLE feedforward
        self.kernel.data.qfrc_applied[:] = 0.0
        self.kernel.data.qfrc_applied[:6] = self.wrapper.nle

        # 6. 데이터 로깅
        if self.logger is not None:
            self.logger.log(
                times=t,
                target_pos=self.target_pos,
                actual_pos=self.current_ee_pos,
                target_vel=self.target_twist[:3],
                actual_vel=self.current_twist[:3],
                q=self.kernel.data.qpos[:6],
                dq=self.kernel.data.qvel[:6],
                ddq=self.kernel.data.qacc[:6],
            )

    def on_render(self, t, visualizer):
        visualizer.draw_axes(self.current_ee_pos, self.current_ee_R, size=0.2)
        visualizer.draw_axes(self.target_pos, self.target_R, size=0.2)
        visualizer.draw_connected_path(wp_pos, rgba=(0, 1, 0, 0.5), thickness=0.003)
        visualizer.draw_trajectory(wp_pos, radius=0.005, rgba=(1, 0, 0, 1))

상태 컨테이너 호출, wrapper 호출, 커널 호출, 제어기 호출, 초기 ee SE3, SO3, x, y, z 추출 (피노키오)

In [ ]:
state    = FixedBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(1, wrapper.model.njoints)] # fixed base는 1부터, 0는 world
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
controller = TaskSpaceController(wrapper)   

print("✅ 시스템 호출 완료")


q_init = kernel.q_keyframe
dq_init = np.zeros(wrapper.nv)
wrapper.update_model(q_init, dq_init)

oM_ee_init = wrapper.oM_ee.copy()
init_ee_pos = oM_ee_init.translation.copy()
init_ee_R = oM_ee_init.rotation.copy()

print(f"초기 자세 (keyframe): \n{q_init}")
print(f"초기 end-effector 위치 x,y,z: \n{init_ee_pos}")
print(f"초기 end-effector 방향 SO(3): \n{init_ee_R}")

In [ ]:
def generate_waypoints(init_site_pos, traj_type=''):

    if traj_type == 'rectangle':
        width = 0.25
        height = 0.25
       
        # X-Z에 대한 사각형
        point1 = init_site_pos
        point2 = point1 + np.array([width, 0.0, 0.0])   
        point3 = point2 + np.array([0.0, 0.0, -height]) 
        point4 = point3 + np.array([-width, 0.0, 0.0])  
        point5 = point1

        return np.vstack((point1, point2, point3, point4, point5))
    
    elif traj_type == 'circle':
        radius = 0.25
        num_points = 40
        
        center = init_site_pos - np.array([radius, 0.0, 0.0])
        theta = np.linspace(0, 2 * np.pi, num_points)

        waypoints = np.zeros((num_points, 3))
        waypoints[:, 0] = center[0] + radius * np.cos(theta) 
        waypoints[:, 1] = init_site_pos[1]                   
        waypoints[:, 2] = center[2] + radius * np.sin(theta) 

        return waypoints

In [ ]:
# 목표 위치 설정
'''
wp_pos = [init_ee_pos,
          init_ee_pos + np.array([-0.2, 0.2, -0.1]),
          init_ee_pos + np.array([0.1, 0.0, -0.25]),
          init_ee_pos + np.array([-0.0, -0.1, -0.3]),
          init_ee_pos + np.array([-0.1, 0.2, -0.1]),
          init_ee_pos + np.array([-0.2, -0.1, -0.1]),
          init_ee_pos + np.array([0.4, 0.3, 0.0])]
'''
wp_pos = generate_waypoints(init_ee_pos, traj_type='rectangle')
    
# 목표 R 설정 
'''
wp_rot = [
        init_ee_R,
        init_ee_R @ rpy.rpyToMatrix(0,       0,      np.pi/4),
        init_ee_R @ rpy.rpyToMatrix(0,  np.pi/6,          0),
        init_ee_R @ rpy.rpyToMatrix(0,       0,      np.pi/2),
        init_ee_R @ rpy.rpyToMatrix(np.pi/6,       np.pi/6,  np.pi/2),
        init_ee_R @ rpy.rpyToMatrix(0,       np.pi/6,  0),
        init_ee_R @ rpy.rpyToMatrix(np.pi/6,       0,      0),]
'''
wp_rot = [init_ee_R] * len(wp_pos)

# 경로 생성
traj = TrajectoryGenerator(
    wp_list=wp_pos,
    R_list=wp_rot,
    target_speed=0.05
)

In [ ]:
# OSQP 솔버 setup (sparsity pattern 만 잡으면 됨)
n  = wrapper.nv          # 6 (UR5e)
m  = n                   # 변수 = dq, box constraint n개
DT = 1/100               # ctrl_hz=100 과 일치 — orchestrator 의 1-step 적분 dt

solver = OSQP_Solver(n=n, m=m)

# 관절 한계 — Pinocchio model에서 직접 (wrapper에는 별도 노출 없음)
dq_max  = wrapper.model.velocityLimit
q_lower = wrapper.model.lowerPositionLimit
q_upper = wrapper.model.upperPositionLimit

# 초기 setup — 매 step 의 J.T@J + λ²I 가 dense (n×n 전부 채움)이므로
# placeholder 도 dense 한 패턴이어야 함. eye(n) 으로 잡으면 upper-tri nnz=n 이라
# 이후 update 시 nnz 가 n(n+1)/2 로 늘어 sparsity 위반.
A_init = np.eye(n)                              # 변수=dq, box → identity (sparsity 고정)
l_init = -dq_max
u_init =  dq_max
H_init = np.ones((n, n))                        # dense pattern (값은 update 로 덮어씀)
q_init = np.zeros(n)
solver.setup(H_init, q_init, A_init, l_init, u_init)

print(f"solver setup OK. n={n}, m={m}")
print(f"dq_max  = {dq_max}")
print(f"q_lower = {q_lower}")
print(f"q_upper = {q_upper}")

Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [ ]:
# 실행 직전 초기화
kernel.reset_to_keyframe()
logger = DataLogger()

# 오케스트레이터 및 스케줄러 실행 (NLE feedforward 항상 인가)
orch  = RobotOrchestrator(kernel, wrapper, controller, traj, solver, DT, logger)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=100, render_hz=60)
sched.run(on_control=orch.on_control, on_render=orch.on_render)

In [ ]:
# 1. 로거에서 데이터 일괄 추출 (Numpy Array 형태)
times = logger.get('times')
target_pos = logger.get('target_pos')
actual_pos = logger.get('actual_pos')
target_vel = logger.get('target_vel')
actual_vel = logger.get('actual_vel')
q = logger.get('q')
dq = logger.get('dq')
ddq = logger.get('ddq')

# 2. 플로팅 함수 호출
# 2-1. 3D 궤적 (실제 이동 경로와 waypoints 비교)
plot_helpers.plot_3d_trajectory(
    pos_log=actual_pos, 
    waypoints=wp_pos, 
    title="UR5e EE 3D Trajectory"
)

# 2-2. X, Y, Z 위치 추종 오차
plot_helpers.plot_ee_tracking(
    times=times, 
    desired=target_pos, 
    actual=actual_pos
)

# 2-3. 선속도 추종 성능 (X, Y, Z 각각 다차원 plot 지원됨)
plot_helpers.plot_velocity_tracking(
    times=times, 
    des_vel=target_vel, 
    act_vel=actual_vel, 
    title="EE Linear Velocity Tracking"
)

# 2-4. 관절 위치 및 속도 상태
plot_helpers.plot_joint_state(
    times=times, 
    q=q, 
    dq=dq, 
    ddq=ddq,
    joint_names=joint_names[:6], # UR5e 관절 이름
    title="UR5e Joint States"
)

# 3. 모든 창이 닫힐 때까지 대기 (Jupyter나 스크립트 종료 방지)
plot_helpers.hold_until_all_fig_closed()

In [ ]:
import matplotlib.pyplot as plt

err_mm = (target_pos - actual_pos) * 1000.0    # m → mm

plt.figure(figsize=(7, 3))
plt.plot(times, err_mm[:, 0], color='red',  linewidth=1.2, label='X error')
plt.plot(times, err_mm[:, 2], color='blue', linewidth=1.2, label='Z error')
plt.axhline(0, color='k', linestyle='--', linewidth=0.7)
plt.xlabel('Time [s]')
plt.ylabel('Position Error [mm]')
plt.title('Position Error')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 추종 오차 max/min/RMS 통계 + 그래프
err = target_pos - actual_pos                       # (N, 3)
stats = plot_helpers.summarize_tracking_error(
    times, err,
    axes=('X', 'Y', 'Z'),
    plot_axes=('X', 'Z'),                            # 그래프에 표시할 축 선택
    title="Position Error (max/min)",
)